# import libraries

In [1]:
import os
import re
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from collections import defaultdict
import random

# set configs

In [2]:
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
IMAGE_DIR = os.path.join(BASE_DIR, "..", "data", "Image_Dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "..", "artifacts")
print(BASE_DIR)


Class_Names = {
    1: "buffalo",
    2: "elephant",
    3: "rhino",
    4: "zebra"
}
SAMPLE_COUNT = 5                   # random samples shown per class in the grid
RANDOM_SEED  = 42

c:\Users\USER-PC\Documents\Laptop\Laptop\Belgium Year 3\MLG 382\Project 1\E-Commerce-Image-Classification\notebooks


In [3]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
PALETTE = sns.color_palette("Set2", 4)

# 1.function to Load metadata

In [4]:
def parse_filename(fname):
    """Extract class label and photo index from 'y (x).ext' filenames."""
    match = re.match(r"^(\d+)\s*\((\d+)\)", fname)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

def load_metadata(directory):
    records = []
    for fname in sorted(os.listdir(directory)):
        label, idx = parse_filename(fname)
        if label is None or label not in Class_Names:
            continue
        fpath = os.path.join(directory, fname)
        try:
            with Image.open(fpath) as img:
                w, h     = img.size
                mode     = img.mode
                arr      = np.array(img.convert("RGB")).astype(np.float32)
                gray_arr = np.array(img.convert("L")).astype(np.float32)
 
            # keeps the array of images in memory
            records.append({
                "path"       : fpath,
                "filename"   : fname,
                "label"      : label,
                "class_name" : Class_Names[label],
                "photo_idx"  : idx,
                "width"      : w,
                "height"     : h,
                "aspect"     : round(w / h, 4),
                "file_bytes" : os.path.getsize(fpath),
                "mode"       : mode,
                "mean_r"     : arr[:, :, 0].mean(),
                "mean_g"     : arr[:, :, 1].mean(),
                "mean_b"     : arr[:, :, 2].mean(),
                "std_r"      : arr[:, :, 0].std(),
                "std_g"      : arr[:, :, 1].std(),
                "std_b"      : arr[:, :, 2].std(),
                "brightness" : gray_arr.mean(),
                "contrast"   : gray_arr.std(),
                "arr"        : arr,          
            })
        except Exception as e:
            print(f"  [skip] {fname}: {e}")
 
    print(f"Loaded {len(records)} images across {len(set(r['label'] for r in records))} classes.\n")
    return records

# 2.function to display the class distribution

In [5]:
def plot_class_distribution(records):
    from collections import Counter
    counts = Counter(r["class_name"] for r in records)
    labels = [Class_Names[i] for i in sorted(Class_Names)]
    values = [counts.get(n, 0) for n in labels]
 
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(labels, values, color=PALETTE, edgecolor="white", linewidth=1.2)
    ax.bar_label(bars, padding=4, fontsize=11, fontweight="bold")
    ax.set_title("Class Distribution", fontsize=14, fontweight="bold")
    ax.set_ylabel("Number of Images")
    ax.set_xlabel("Animal Class")
    ax.set_ylim(0, max(values) * 1.2)
    sns.despine()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "1_class_distribution.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[1] Class distribution  →  {path}")
    for l, v in zip(labels, values):
        print(f"    {l}: {v} images")
    print()


# 3.Image dimension comparrison

In [6]:
def plot_dimensions(records):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
 
    by_class = defaultdict(list)
    for r in records:
        by_class[r["class_name"]].append(r)
 
    # Scatter: width vs height, coloured by class
    ax = axes[0]
    for i, (cname, recs) in enumerate(sorted(by_class.items())):
        ws = [r["width"]  for r in recs]
        hs = [r["height"] for r in recs]
        ax.scatter(ws, hs, label=cname, alpha=0.65, s=40, color=PALETTE[i])
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    ax.set_title("Width vs Height")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
 
    # Histogram: widths
    ax = axes[1]
    for i, (cname, recs) in enumerate(sorted(by_class.items())):
        ws = [r["width"] for r in recs]
        ax.hist(ws, bins=20, alpha=0.6, label=cname, color=PALETTE[i], edgecolor="none")
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Count")
    ax.set_title("Width Distribution")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
 
    # Histogram: heights
    ax = axes[2]
    for i, (cname, recs) in enumerate(sorted(by_class.items())):
        hs = [r["height"] for r in recs]
        ax.hist(hs, bins=20, alpha=0.6, label=cname, color=PALETTE[i], edgecolor="none")
    ax.set_xlabel("Height (px)")
    ax.set_ylabel("Count")
    ax.set_title("Height Distribution")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
 
    plt.suptitle("Image Dimensions", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "2_image_dimensions.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[2] Image dimensions    →  {path}\n")

# 4. Aspect ratio comparison

In [7]:
def plot_aspect_ratio(records):
    fig, ax = plt.subplots(figsize=(8, 4))
    by_class = defaultdict(list)
    for r in records:
        by_class[r["class_name"]].append(r["aspect"])
 
    for i, (cname, aspects) in enumerate(sorted(by_class.items())):
        ax.hist(aspects, bins=25, alpha=0.65, label=cname, color=PALETTE[i], edgecolor="none")
 
    ax.axvline(1.0, color="black", linestyle="--", linewidth=1.2, label="Square (1:1)")
    ax.set_xlabel("Aspect Ratio  (width / height)")
    ax.set_ylabel("Count")
    ax.set_title("Aspect Ratio Distribution", fontsize=14, fontweight="bold")
    ax.legend(fontsize=9)
    sns.despine()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "3_aspect_ratio.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[3] Aspect ratio        →  {path}\n")

# 5. File size comparrison

In [8]:
def plot_file_size(records):
    fig, ax = plt.subplots(figsize=(7, 4))
    by_class = defaultdict(list)
    for r in records:
        by_class[r["class_name"]].append(r["file_bytes"] / 1024)   # KB
 
    labels  = sorted(by_class.keys())
    data    = [by_class[l] for l in labels]
    colors  = [PALETTE[i] for i in range(len(labels))]
 
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
 
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels)
    ax.set_ylabel("File Size (KB)")
    ax.set_title("File Size Distribution per Class", fontsize=14, fontweight="bold")
    sns.despine()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "4_file_size.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"[4] File size           →  {path}\n")

#  6. Color comparison

#### compares the brightness, the contrast, and saturation of the images

In [9]:
def plot_color_stats(records):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    channels  = ["mean_r", "mean_g", "mean_b"]
    ch_labels = ["Red", "Green", "Blue"]
    ch_colors = ["#e74c3c", "#2ecc71", "#3498db"]
 
    class_labels = sorted(set(r["class_name"] for r in records))
    x            = np.arange(len(class_labels))
    width        = 0.25
 
    # Mean per channel
    ax = axes[0]
    for i, (ch, label, color) in enumerate(zip(channels, ch_labels, ch_colors)):
        means = [np.mean([r[ch] for r in records if r["class_name"] == cl])
                 for cl in class_labels]
        ax.bar(x + i * width, means, width, label=label, color=color, alpha=0.8, edgecolor="white")
    ax.set_xticks(x + width)
    ax.set_xticklabels(class_labels)
    ax.set_ylabel("Mean Pixel Value (0–255)")
    ax.set_title("Mean RGB per Class")
    ax.legend()
    sns.despine(ax=ax)
 
    # Brightness & contrast
    ax = axes[1]
    brightness = [np.mean([r["brightness"] for r in records if r["class_name"] == cl])
                  for cl in class_labels]
    contrast   = [np.mean([r["contrast"]   for r in records if r["class_name"] == cl])
                  for cl in class_labels]
    ax.bar(x - 0.2, brightness, 0.35, label="Brightness (mean)", color="#f39c12", alpha=0.85, edgecolor="white")
    ax.bar(x + 0.2, contrast,   0.35, label="Contrast (std)",    color="#8e44ad", alpha=0.85, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels)
    ax.set_ylabel("Pixel Value")
    ax.set_title("Brightness & Contrast per Class")
    ax.legend()
    sns.despine(ax=ax)
 
    plt.suptitle("Color & Brightness Statistics", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "5_color_stats.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[5] Color statistics    →  {path}\n")

#  7. SAMPLE IMAGE GRID

In [10]:
def plot_sample_grid(records):
    by_class = defaultdict(list)
    for r in records:
        by_class[r["label"]].append(r)
 
    n_classes = len(Class_Names)
    n_cols    = SAMPLE_COUNT
    fig       = plt.figure(figsize=(n_cols * 2.5, n_classes * 2.8))
    gs        = gridspec.GridSpec(n_classes, n_cols, figure=fig,
                                  hspace=0.4, wspace=0.1)
 
    for row, label in enumerate(sorted(Class_Names.keys())):
        pool    = by_class.get(label, [])
        samples = random.sample(pool, min(n_cols, len(pool)))
 
        for col, rec in enumerate(samples):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(rec["arr"].astype(np.uint8))
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(Class_Names[label], fontsize=11,
                              fontweight="bold", labelpad=6)
                ax.yaxis.set_label_position("left")
                ax.yaxis.label.set_visible(True)
            ax.set_title(f"#{rec['photo_idx']}", fontsize=8, pad=2)
 
    fig.suptitle("Random Sample Grid  (rows = classes, cols = samples)",
                 fontsize=13, fontweight="bold", y=1.01)
    path = os.path.join(OUTPUT_DIR, "6_sample_grid.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[6] Sample grid         →  {path}\n")

# 8. print out of stat summary

In [11]:
def print_summary(records):
    print("=" * 55)
    print("  DATASET SUMMARY")
    print("=" * 55)
    total = len(records)
    print(f"  Total images   : {total}")
    print(f"  Classes        : {len(Class_Names)}")
    widths  = [r["width"]  for r in records]
    heights = [r["height"] for r in records]
    print(f"  Width  range   : {min(widths)} – {max(widths)} px")
    print(f"  Height range   : {min(heights)} – {max(heights)} px")
    aspects = [r["aspect"] for r in records]
    print(f"  Aspect ratio   : {min(aspects):.2f} – {max(aspects):.2f}")
    sizes   = [r["file_bytes"] / 1024 for r in records]
    print(f"  File size      : {min(sizes):.1f} – {max(sizes):.1f} KB")
    print()
    from collections import Counter
    counts = Counter(r["class_name"] for r in records)
    for cname, cnt in sorted(counts.items()):
        balance = cnt / total * 100
        print(f"  {cname:<15}: {cnt:>4} images  ({balance:.1f}%)")
    print("=" * 55)

# 9.Save the graphs

In [12]:
if __name__ == "__main__":
    print(f"\nLoading images from: {os.path.abspath(IMAGE_DIR)}\n")
    records = load_metadata(IMAGE_DIR)
 
    if not records:
        print("No images found. Check IMAGE_DIR and filenames match 'y (x).ext'.")
    else:
        print_summary(records)
        print(f"\nGenerating plots → {os.path.abspath(OUTPUT_DIR)}\n")
        plot_class_distribution(records)
        plot_dimensions(records)
        plot_aspect_ratio(records)
        plot_file_size(records)
        plot_color_stats(records)
        plot_sample_grid(records)
        print("\nAll done! Open the eda_outputs/ folder to view your plots.")


Loading images from: c:\Users\USER-PC\Documents\Laptop\Laptop\Belgium Year 3\MLG 382\Project 1\E-Commerce-Image-Classification\data\Image_Dataset

Loaded 1504 images across 4 classes.

  DATASET SUMMARY
  Total images   : 1504
  Classes        : 4
  Width  range   : 99 – 1920 px
  Height range   : 86 – 1920 px
  Aspect ratio   : 0.64 – 3.22
  File size      : 2.0 – 541.1 KB

  buffalo        :  376 images  (25.0%)
  elephant       :  376 images  (25.0%)
  rhino          :  376 images  (25.0%)
  zebra          :  376 images  (25.0%)

Generating plots → c:\Users\USER-PC\Documents\Laptop\Laptop\Belgium Year 3\MLG 382\Project 1\E-Commerce-Image-Classification\artifacts

[1] Class distribution  →  c:\Users\USER-PC\Documents\Laptop\Laptop\Belgium Year 3\MLG 382\Project 1\E-Commerce-Image-Classification\notebooks\..\artifacts\1_class_distribution.png
    buffalo: 376 images
    elephant: 376 images
    rhino: 376 images
    zebra: 376 images

[2] Image dimensions    →  c:\Users\USER-PC\Docum